<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import numpy as np
import torch
import tensorly as tl
from tensorly.decomposition import parafac
tl.set_backend('pytorch')
import sparse

# -------------------------------
# 1. PyTorch-based BPTF (your version)
# -------------------------------
# Import your PyTorch-based BPTF model (adjust filename as needed)
from own_implementation import BPTF as BPTF_torch

# Set random seeds for reproducibility
np.random.seed(0)
torch.manual_seed(0)

dimension = 100
# Use a 100x100x100 tensor (1,000,000 elements)
expected_shape = (dimension, dimension, dimension)

# Create a tensor from a Poisson distribution (counts) and a matching mask; ensure types match
data_torch_np = np.random.poisson(lam=5, size=expected_shape)
data_torch = torch.tensor(data_torch_np, dtype=torch.float64)
mask_torch = torch.ones(expected_shape, dtype=torch.float64)

# Instantiate and fit the PyTorch-based BPTF model
model_torch = BPTF_torch(data_shape=expected_shape, n_components=3, alpha=0.1, device="cpu")
model_torch.fit(data_torch, mask=mask_torch, max_iter=50, tol=1e-4, verbose=True)
reconstruction_torch = model_torch.reconstruct(mask=mask_torch, style='arithmetic')
frobenius_diff_torch = torch.norm(data_torch - reconstruction_torch, p='fro').item()
print("PyTorch BPTF reconstruction Frobenius norm difference:", frobenius_diff_torch)

# -------------------------------
# 2. TensorLy CP Decomposition
# -------------------------------
# Use TensorLy's parafac for CP decomposition (same rank as n_components)
cp_decomp = parafac(data_torch, rank=3, n_iter_max=100, init='svd')
reconstruction_cp = tl.cp_to_tensor(cp_decomp)
frobenius_diff_cp = torch.norm(data_torch - reconstruction_cp, p='fro').item()
print("TensorLy CP decomposition Frobenius norm difference:", frobenius_diff_cp)

# -------------------------------
# 3. NumPy-based BPTF (Aaron's original implementation)
# -------------------------------
# Import Aaron's BPTF (which uses NumPy/sparse.COO); 
# ensure that the bptf package is in your PYTHONPATH.
from bptf import BPTF as BPTF  # :contentReference[oaicite:2]{index=2}
import bptf

# Create the same data as a NumPy array and a corresponding binary mask
data_np = np.random.poisson(lam=5, size=expected_shape).astype(int)
data_np = sparse.COO.from_numpy(data_np.copy())
mask_np = np.ones(expected_shape, dtype=int)
mask_np = sparse.COO.from_numpy(mask_np.copy())

def _check_mode(self, m):
    assert np.isfinite(np.asarray(self.E_DK_M[m])).all()
    assert np.isfinite(np.asarray(self.G_DK_M[m])).all()
    assert np.isfinite(np.asarray(self.shp_DK_M[m])).all()
    assert np.isfinite(np.asarray(self.rte_DK_M[m])).all()

bptf.BPTF._check_mode = _check_mode

# Instantiate and fit the NumPy-based BPTF model.
# Note: This version uses its own preprocess() function and can work with sparse.COO.
model_np = BPTF(data_shape=data_np.shape, n_components=3, alpha=0.1)
model_np.fit(data_np, mask=mask_np, max_iter=50, verbose=True)

# Reconstruct using arithmetic expectation
reconstruction_np = model_np.reconstruct(mask=mask_np, fill_value=0, drop_diag=False, style='arithmetic')
frobenius_diff_np = np.linalg.norm(data_np - reconstruction_np, ord='fro')
print("NumPy BPTF reconstruction Frobenius norm difference:", frobenius_diff_np)


  0%|                                                                                                                                                                                     | 0/50 [00:00<?, ?it/s]

ELBO = -53594944.462336324, change = -15.289666719934445, time taken = 0.06035208702087402:   0%|                                                                                         | 0/50 [00:00<?, ?it/s]

ELBO = -163031245.66241628, change = -2.0419146301566924, time taken = 0.028502464294433594:   0%|                                                                                        | 0/50 [00:00<?, ?it/s]

ELBO = -163031196.3023333, change = 3.027645576982159e-07, time taken = 0.02746105194091797:   0%|                                                                                        | 0/50 [00:00<?, ?it/s]

ELBO = -163031196.3023333, change = 3.027645576982159e-07, time taken = 0.02746105194091797:   6%|████▊                                                                           | 3/50 [00:00<00:01, 25.11it/s]

...

ELBO = -163030572.61736834, change = 3.0578373758119716e-06, time taken = 0.024883508682250977: 100%|████████████████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 37.73it/s]

ELBO = -163030572.61736834, change = 3.0578373758119716e-06, time taken = 0.024883508682250977: 100%|████████████████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 35.12it/s]

PyTorch BPTF reconstruction Frobenius norm difference: 2235.972537612717


TensorLy CP decomposition Frobenius norm difference: 2233.6509599615238


TypeError: ufunc 'isfinite' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''